In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import col, current_timestamp

In [0]:
dbutils.widgets.text("catalogo", "proyecto_olist")
dbutils.widgets.text("esquema_source", "silver")
dbutils.widgets.text("esquema_sink", "golden")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

In [0]:
df_silver_orders = spark.table(f"{catalogo}.{esquema_source}.orders")
df_silver_items = spark.table(f"{catalogo}.{esquema_source}.order_items")

In [0]:
fact_orders = df_silver_orders.join(df_silver_items, "order_id", "inner") \
    .select(
        col("order_id"),
        col("customer_id"), 
        col("product_id"),
        col("order_status"),
        col("order_purchase_timestamp"),
        col("price").alias("amount"),
        col("freight_value").alias("shipping_cost")
    ).withColumn("gold_insert_date", current_timestamp())

In [0]:
fact_orders.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.fact_orders")

In [0]:
dbutils.widgets.removeAll()